# Member 2: Research Questions Q2 and Q10
**DEV Milestone 1 · NYC Housing Maintenance Complaints (H6) × SIRTA Serious Crashes (T4), 2022–2025**

| # | Type | Question | Data |
|---|---|---|---|
| Q2 | Cross-domain | Is the type of housing problem more closely linked to serious crashes than the total volume? | `integrated_cd_year.csv` |
| Q10 | Transport | At what hours and days are serious crashes most deadly? | `crashes.parquet` |

Each question has: the question and why it matters, the chart (a `build_qN` function returning a Plotly
figure, reused unchanged on the website), the numbers behind the answer, and the answer.
Both charts use only the small website files, so the notebook and the website show identical results.

In [4]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

FILES = ['integrated_cd_year.csv', 'crashes.parquet']
SEARCH = ['/content',
          '/content/drive/MyDrive/DEV_project_data/web_data',
          '/content/drive/MyDrive/DEV_project_data']
WEB = next((p for p in SEARCH if all(os.path.exists(f'{p}/{f}') for f in FILES)), None)

if WEB is None:
    from google.colab import files
    print('Please upload integrated_cd_year.csv and crashes.parquet')
    for name, content in files.upload().items():
        target = next((f for f in FILES if f.split('.')[0] in name and name.endswith(f.split('.')[1])), name)
        with open(f'/content/{target}', 'wb') as fh:
            fh.write(content)
    WEB = '/content'

df_integrated = pd.read_csv(f'{WEB}/integrated_cd_year.csv')
df_crashes    = pd.read_parquet(f'{WEB}/crashes.parquet')
print(f'Loaded from {WEB}: {len(df_integrated)} district-years, {len(df_crashes):,} crashes')

Loaded from /content: 236 district-years, 11,953 crashes


## Q2. Is the type of housing problem more closely linked to serious crashes than the total volume?

**Question:** Are some *types* of housing problems (for example unsanitary conditions, heat/hot water or
water leaks) more strongly associated with serious crashes than the total number of housing problems?

**Why it matters:** across district-years, total housing problems and serious crashes are only weakly
correlated (r = 0.15, Section 5.1). Total volume mostly reflects how large and dense a district is. If the
*mix* of problems tracks crashes more closely, it points to a shared neighbourhood profile (for example
older, poorly maintained areas with dangerous streets) that HPD and DOT could target together.

**Method:** for each housing category (`cat_*` columns plus `heat_problems`, since `cat_heat_hot_water` was
dropped as a duplicate in 5.2), compute its **share of the district's housing problems**, so big districts
do not dominate. Then correlate each share with serious crashes across the 236 district-years and compare
with the correlation for total volume (grey reference bar). Dotted lines mark the smallest |r| that is
statistically different from zero at the 5% level.

In [5]:
# ============================================================
# Q2: Is the type of housing problem more closely linked to serious crashes than the total volume?
# ============================================================
def category_label(col):
    """'cat_unsanitary_condition' → 'Unsanitary condition'; 'heat_problems' → 'Heat/hot water'."""
    if col == 'heat_problems':
        return 'Heat/hot water'
    return col.replace('cat_', '').replace('_', ' ').capitalize().replace('Door window', 'Door/window').replace('Paint plaster', 'Paint/plaster').replace('Flooring stairs', 'Flooring/stairs')


def problem_type_corr_table(df_integrated):
    """Correlation of each category's share of housing problems with serious crashes, across district-years."""
    d = df_integrated
    # cat_heat_hot_water was dropped in 5.2 (identical to heat_problems), so heat comes from heat_problems
    cols = [c for c in d.columns if c.startswith('cat_')] + ['heat_problems']
    total = d['housing_problems'].replace(0, np.nan)

    rows = []
    for c in cols:
        share = d[c] / total * 100
        if not share.std() > 0:   # constant share (e.g. always 0) has no correlation
            continue
        r = share.corr(d['crashes'])
        if pd.notna(r):   # a category with no variation (e.g. after filtering) has no correlation
            rows.append({'measure': category_label(c), 'r': r, 'problems': int(d[c].sum()),
                         'avg_share_%': share.mean(), 'kind': 'Positive' if r >= 0 else 'Negative'})

    # Reference bar: total volume (computed, not hard-coded, so it follows the website filters)
    rows.append({'measure': 'TOTAL VOLUME (reference)', 'r': d['housing_problems'].corr(d['crashes']),
                 'problems': int(d['housing_problems'].sum()), 'avg_share_%': 100.0,
                 'kind': 'Total volume (reference)'})

    table = pd.DataFrame(rows).sort_values('r').reset_index(drop=True)
    n = len(d)
    # Smallest |r| that is statistically different from 0 at the 5% level (t ≈ 1.97 for n ≈ 236)
    r_crit = 1.97 / np.sqrt(n - 2 + 1.97 ** 2) if n > 3 else np.nan
    return table, n, r_crit


def build_q2(df_integrated):
    """Sorted horizontal bar: correlation of each problem-type share with serious crashes."""
    table, n, r_crit = problem_type_corr_table(df_integrated)

    fig = px.bar(
        table, x='r', y='measure', orientation='h', color='kind',
        color_discrete_map={'Positive': '#b5562f', 'Negative': '#2557a7',
                            'Total volume (reference)': '#8c8c8c'},
        custom_data=['problems', 'avg_share_%'],
        labels={'r': 'Correlation with serious crashes (Pearson r)', 'measure': 'Housing problem type (share of all problems)',
                'kind': ''},
        title=f'Q2. Problem type vs serious crashes: correlation of each category share with crashes '
              f'({n} district-years)')
    fig.update_traces(hovertemplate='<b>%{y}</b><br>r = %{x:.3f}<br>Problems: %{customdata[0]:,}'
                                    '<br>Average share: %{customdata[1]:.1f}%<extra></extra>')

    fig.add_vline(x=0, line_color='black', line_width=1)
    if pd.notna(r_crit):
        for x in (-r_crit, r_crit):
            fig.add_vline(x=x, line_dash='dot', line_color='gray')
        fig.add_annotation(x=r_crit, y=1.07, yref='paper', showarrow=False, xanchor='left',
                           text=f'|r| < {r_crit:.2f}: not significant', font=dict(size=10, color='gray'))

    lim = max(0.3, table['r'].abs().max() * 1.2)
    fig.update_layout(template='plotly_white', height=max(480, 30 * len(table)), xaxis_range=[-lim, lim],
                      yaxis=dict(categoryorder='array', categoryarray=table['measure'].tolist()),
                      legend=dict(orientation='h', y=-0.15))
    return fig


build_q2(df_integrated).show()

In [6]:
# ============================================================
# Q2: numbers for the answer
# ============================================================
q2_table, q2_n, q2_r_crit = problem_type_corr_table(df_integrated)
q2_r_total = q2_table.loc[q2_table['measure'] == 'TOTAL VOLUME (reference)', 'r'].iloc[0]
q2_cats = q2_table[q2_table['measure'] != 'TOTAL VOLUME (reference)']

print(f'District-years: {q2_n} | significance threshold |r| ≥ {q2_r_crit:.2f}')
print(f'Total volume vs crashes: r = {q2_r_total:.3f}')
print(f"Categories with |r| above total volume: {(q2_cats['r'].abs() > abs(q2_r_total)).sum()} of {len(q2_cats)}")
print(f"Significant categories: {(q2_cats['r'].abs() >= q2_r_crit).sum()} of {len(q2_cats)}")
display(q2_table.sort_values('r', ascending=False)
        .assign(r=lambda t: t['r'].round(3), **{'avg_share_%': lambda t: t['avg_share_%'].round(1)})
        [['measure', 'r', 'avg_share_%', 'problems']])

# Context for the answer: which boroughs have high shares of the top positive / top negative category
q2_top_pos = q2_cats.iloc[-1]['measure']
q2_top_neg = q2_cats.iloc[0]['measure']
_share = lambda name: [c for c in df_integrated.columns if c.startswith('cat_') or c == 'heat_problems'
                       if category_label(c) == name][0]
q2_boro = (df_integrated.groupby('borough')[[_share(q2_top_pos), _share(q2_top_neg), 'housing_problems', 'crashes']].sum())
q2_boro[f'{q2_top_pos} share %'] = (q2_boro[_share(q2_top_pos)] / q2_boro['housing_problems'] * 100).round(1)
q2_boro[f'{q2_top_neg} share %'] = (q2_boro[_share(q2_top_neg)] / q2_boro['housing_problems'] * 100).round(1)
q2_boro['crashes per year'] = (q2_boro['crashes'] / df_integrated['year'].nunique()).round(0)
display(q2_boro[[f'{q2_top_pos} share %', f'{q2_top_neg} share %', 'crashes per year']])

District-years: 236 | significance threshold |r| ≥ 0.13
Total volume vs crashes: r = 0.152
Categories with |r| above total volume: 2 of 14
Significant categories: 2 of 14


,measure,r,avg_share_%,problems
14,TOTAL VOLUME (reference),0.152,100.0,2657915
13,Electric,0.124,4.9,125987
12,Appliance,0.120,3.4,95670
11,Plumbing,0.088,10.5,284444
10,Paint/plaster,0.047,9.6,278048
9,Unsanitary condition,0.046,20.1,520732
8,Water leak,0.029,6.3,174831
7,General,-0.002,4.8,128553
6,Door/window,-0.002,7.9,216256
5,Flooring/stairs,-0.024,4.2,121471


,Electric share %,Elevator share %,crashes per year
borough,,,
BRONX,4.6,0.3,540.0
BROOKLYN,5.1,0.2,840.0
MANHATTAN,4.1,0.3,543.0
QUEENS,5.3,0.3,663.0
STATEN ISLAND,5.9,0.1,124.0


### Q2 Answer

**No. No single type of housing problem is more strongly *positively* associated with serious crashes
than the total volume (r = 0.15).** The only categories with a stronger association than volume are
*negative*: districts where a larger share of problems concern **elevators** or the **outside of the
building** tend to have **fewer** serious crashes.

| Measure (share of all problems) | r with serious crashes | Average share | Problems 2022–2025 |
|---|---|---|---|
| Total volume (reference) | **+0.152** | 100% | 2,657,915 |
| Electric | +0.124 | 4.9% | 125,987 |
| Appliance | +0.120 | 3.4% | 95,670 |
| Plumbing | +0.088 | 10.5% | 284,444 |
| Heat/hot water | −0.079 | 24.2% | 608,708 |
| Outside building | −0.166 | 0.3% | 7,409 |
| Elevator | **−0.186** | 0.3% | 7,335 |

**Findings:**
- **Total volume stays the strongest positive signal.** The strongest positive categories, **electric**
  (r = 0.124) and **appliance** (r = 0.120), come close but are weaker than volume and below the
  significance threshold (|r| ≥ 0.13 for 236 district-years).
- **Only 2 of 14 categories pass the threshold, and both are negative:** elevator (r = −0.186) and outside
  building (r = −0.166). Districts in the top quarter for elevator share average **38.8 serious crashes per
  year**, compared with **45.7** in the bottom quarter. A high elevator share may mark districts with more
  large apartment buildings, but this data cannot confirm that.
- **The biggest categories show almost no link:** unsanitary condition (20.1% of problems, r = 0.046) and
  heat/hot water (24.2%, r = −0.079) are both close to zero.
- Every correlation is **weak** (|r| < 0.2): the problem mix explains at most about **3.5%** of the variation
  in serious crashes between district-years. Correlation does not mean that one causes the other.

**Benefit to NYC:** the type of housing complaint is **not a useful early-warning signal** for road danger,
so HPD complaint data cannot stand in for traffic-safety data when DOT chooses where to act. Joint
HPD and DOT targeting should use the districts that are high on *both* measures (Q1) rather than a
particular complaint type.

**Limitations:** the elevator and outside-building categories are very small (~7,400 problems each, 0.3%
of the total), so their shares are noisy. The associations are **not stable over time**: the elevator
correlation is −0.46 in 2022 but only about −0.1 in 2023–2025, and total volume drops from r = 0.35 (2022)
to 0.03 (2025). Both measures are shaped by shared factors such as density, traffic volume and building
type, which are not controlled for. The 236 district-years are the same 59 districts four times, so they
are not fully independent and the significance threshold is approximate.

## Q10. At what hours and days are serious crashes most deadly?

**Question:** At which times of day and days of the week does a serious crash most often end in a death?

**Why it matters:** NYC's Vision Zero programme aims to eliminate traffic deaths. Knowing *when* serious
crashes are most likely to be fatal helps NYPD and DOT target speed enforcement, street lighting and
emergency-response capacity at the right hours, which may not be the hours with the most crashes.

**Method:** all 11,953 serious crashes are grouped by weekday (Monday → Sunday) and **3-hour block**
(00–02h, 03–05h, …), so each cell has enough crashes. The **fatality rate** is fatalities per 100 serious
crashes. Cells with fewer than 30 crashes are left blank (relevant when the website filters one borough or
year). A second heatmap shows the number of crashes, because the busiest times are not necessarily the
deadliest. The hover shows the crash count, fatalities and % of crashes with a death.

In [7]:
# ============================================================
# Q10: At what hours and days are serious crashes most deadly?
# ============================================================
WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HOUR_BLOCKS   = [f'{h:02d}–{h + 2:02d}h' for h in range(0, 24, 3)]   # 00–02h, 03–05h, ..., 21–23h
MIN_CRASHES   = 30   # cells with fewer crashes are left blank (rate too unstable)


def deadly_hours_table(df_crashes):
    """Crashes, fatalities and fatality rate per weekday × 3-hour block."""
    d = df_crashes.dropna(subset=['crash_hour']).copy()
    d['day'] = d['weekday'].astype(str)
    d['block'] = (d['crash_hour'].astype(int) // 3 * 3).map(lambda h: f'{h:02d}–{h + 2:02d}h')
    d['is_fatal'] = d['total_fatalities'] > 0

    t = (d.groupby(['day', 'block'])
          .agg(crashes=('total_fatalities', 'size'), fatalities=('total_fatalities', 'sum'),
               fatal_crashes=('is_fatal', 'sum'))
          .reset_index())
    t['fatality_rate'] = t['fatalities'] / t['crashes'] * 100          # fatalities per 100 crashes
    t['fatal_crash_%'] = t['fatal_crashes'] / t['crashes'] * 100       # % of crashes with ≥ 1 death
    return t


def _grid(t, value):
    """Pivot to a weekday (rows, Mon→Sun) × hour-block (columns) grid."""
    days = [w for w in WEEKDAY_ORDER if w in set(t['day'])]
    return t.pivot(index='day', columns='block', values=value).reindex(index=days, columns=HOUR_BLOCKS)


def build_q10(df_crashes):
    """Heatmap: fatalities per 100 serious crashes, by weekday and 3-hour block (crash count in hover)."""
    t = deadly_hours_table(df_crashes)
    rate = _grid(t, 'fatality_rate')
    crashes = _grid(t, 'crashes').fillna(0)
    rate = rate.where(crashes >= MIN_CRASHES)   # small-sample rule

    fig = px.imshow(
        rate.round(1), x=HOUR_BLOCKS, y=rate.index.tolist(),
        color_continuous_scale='Reds', aspect='auto', text_auto='.1f',
        labels={'x': 'Time of day (3-hour block)', 'y': 'Day of week', 'color': 'Fatalities per 100 crashes'},
        title='Q10. Deadliest hours and days: fatalities per 100 serious crashes')
    fig.update_traces(
        customdata=np.dstack([crashes.values, _grid(t, 'fatalities').fillna(0).values,
                              _grid(t, 'fatal_crash_%').values]),
        hovertemplate='%{y}, %{x}<br>Fatalities per 100 crashes: %{z:.1f}'
                      '<br>Serious crashes: %{customdata[0]:,.0f}<br>Fatalities: %{customdata[1]:,.0f}'
                      '<br>Crashes with a death: %{customdata[2]:.1f}%<extra></extra>')
    fig.update_layout(template='plotly_white', height=440,
                      coloraxis_colorbar=dict(title='Deaths per<br>100 crashes'))
    if (crashes < MIN_CRASHES).any().any():
        fig.add_annotation(text=f'Blank cells: fewer than {MIN_CRASHES} crashes', showarrow=False,
                           x=0, y=-0.18, xref='paper', yref='paper', xanchor='left', font=dict(size=10, color='gray'))
    return fig


def build_q10_counts(df_crashes):
    """Companion heatmap: number of serious crashes (the busiest times are not the deadliest)."""
    t = deadly_hours_table(df_crashes)
    crashes = _grid(t, 'crashes').fillna(0)
    fig = px.imshow(
        crashes, x=HOUR_BLOCKS, y=crashes.index.tolist(),
        color_continuous_scale='Blues', aspect='auto', text_auto=',.0f',
        labels={'x': 'Time of day (3-hour block)', 'y': 'Day of week', 'color': 'Serious crashes'},
        title='Q10. Busiest hours and days: number of serious crashes')
    fig.update_traces(hovertemplate='%{y}, %{x}<br>Serious crashes: %{z:,.0f}<extra></extra>')
    fig.update_layout(template='plotly_white', height=440, coloraxis_colorbar=dict(title='Crashes'))
    return fig


build_q10(df_crashes).show()
build_q10_counts(df_crashes).show()

In [8]:
# ============================================================
# Q10: numbers for the answer
# ============================================================
q10_t = deadly_hours_table(df_crashes)
q10_ok = q10_t[q10_t['crashes'] >= MIN_CRASHES]

overall = df_crashes['total_fatalities'].sum() / len(df_crashes) * 100
print(f"All serious crashes: {len(df_crashes):,} | fatalities: {int(df_crashes['total_fatalities'].sum()):,} "
      f"| {overall:.1f} fatalities per 100 crashes")
print(f'Cells with at least {MIN_CRASHES} crashes: {len(q10_ok)} of {len(q10_t)}')

cols = ['day', 'block', 'crashes', 'fatalities', 'fatality_rate', 'fatal_crash_%']
print('\nDeadliest cells:')
display(q10_ok.nlargest(5, 'fatality_rate')[cols].round(1))
print('Busiest cells:')
display(q10_t.nlargest(5, 'crashes')[cols].round(1))
print('Safest cells (lowest fatality rate):')
display(q10_ok.nsmallest(3, 'fatality_rate')[cols].round(1))

# Night (00–05h) vs rest of the day, and weekend vs weekday
_d = df_crashes.dropna(subset=['crash_hour'])
_night = _d['crash_hour'] < 6
_wknd = _d['weekday'].astype(str).isin(['Saturday', 'Sunday'])
q10_split = pd.DataFrame({
    'crashes':   [(_night).sum(), (~_night).sum(), _wknd.sum(), (~_wknd).sum()],
    'fatalities': [_d.loc[_night, 'total_fatalities'].sum(), _d.loc[~_night, 'total_fatalities'].sum(),
                   _d.loc[_wknd, 'total_fatalities'].sum(), _d.loc[~_wknd, 'total_fatalities'].sum()],
}, index=['00–05h', '06–23h', 'Weekend', 'Weekday'])
q10_split['per 100 crashes'] = (q10_split['fatalities'] / q10_split['crashes'] * 100).round(1)
display(q10_split)

All serious crashes: 11,953 | fatalities: 975 | 8.2 fatalities per 100 crashes
Cells with at least 30 crashes: 56 of 56

Deadliest cells:


,day,block,crashes,fatalities,fatality_rate,fatal_crash_%
1,Friday,03–05h,93,23,24.7,22.6
33,Thursday,03–05h,74,17,23.0,23.0
40,Tuesday,00–02h,99,19,19.2,18.2
17,Saturday,03–05h,184,32,17.4,16.3
41,Tuesday,03–05h,64,11,17.2,17.2


Busiest cells:


,day,block,crashes,fatalities,fatality_rate,fatal_crash_%
5,Friday,15–17h,350,15,4.3,4.3
37,Thursday,15–17h,333,15,4.5,4.5
53,Wednesday,15–17h,326,18,5.5,5.5
6,Friday,18–20h,321,21,6.5,6.5
13,Monday,15–17h,308,16,5.2,5.2


Safest cells (lowest fatality rate):


,day,block,crashes,fatalities,fatality_rate,fatal_crash_%
43,Tuesday,09–11h,211,6,2.8,2.8
29,Sunday,15–17h,244,8,3.3,3.3
5,Friday,15–17h,350,15,4.3,4.3


,crashes,fatalities,per 100 crashes
00–05h,1989,285,14.3
06–23h,9964,690,6.9
Weekend,3440,315,9.2
Weekday,8513,660,7.8


### Q10 Answer

**Serious crashes are most deadly in the early morning (00:00–05:59), above all from 03:00 to 05:59. The
busiest time, the weekday afternoon rush (15:00–17:59), is among the least deadly.**

| Time | Serious crashes | Fatalities | Fatalities per 100 crashes |
|---|---|---|---|
| All crashes, 2022–2025 | 11,953 | 975 | **8.2** |
| Deadliest: Friday 03–05h | 93 | 23 | **24.7** |
| 2nd deadliest: Thursday 03–05h | 74 | 17 | 23.0 |
| 3rd deadliest: Tuesday 00–02h | 99 | 19 | 19.2 |
| Busiest: Friday 15–17h | 350 | 15 | 4.3 |
| 00–05h vs 06–23h | 1,989 vs 9,964 | 285 vs 690 | **14.3 vs 6.9** |

**Findings:**
- **The early morning is the deadliest time of day.** Crashes between 00:00 and 05:59 have **14.3 fatalities
  per 100 crashes, about 2× the rest of the day (6.9)**. They are only **16.6% of serious crashes but 29.2%
  of deaths**. The single worst block is **03:00–05:59** (17.6 per 100 overall), and every day of the week
  has a rate of at least 14 in that block.
- **The deadliest cells are Friday 03–05h (24.7) and Thursday 03–05h (23.0)**, about **3× the average**.
- **The busiest times are not the deadliest:** the afternoon rush 15:00–17:59 has the most crashes (2,122;
  Friday alone 350) but the **lowest fatality rate (4.6 per 100)**, about a quarter of the 03–05h rate.
  Heavier traffic is likely to be slower, which is associated with injuries rather than deaths.
- **Weekends are slightly deadlier** (9.2 per 100 crashes compared with 7.8 on weekdays), mostly because
  weekend nights have many more early-morning crashes (Saturday and Sunday 00–05h: 918 crashes, compared
  with 163–266 for each weekday).

**Benefit to NYC:** for Vision Zero, reducing deaths means targeting the **late night and early morning**,
for example speed-camera enforcement overnight, signal timing and lighting on fast arterial roads, and
emergency-response capacity between midnight and 06:00, rather than only the rush hours when most crashes
happen.

**Limitations:** fatalities are rare (7.7% of serious crashes involve a death), so single cells are noisy:
the weekday 03–05h cells have only 64–93 crashes each, and a few multi-fatality crashes can move them by
several points. The rate does not account for how much traffic is on the road at each hour. Higher
early-morning severity is associated with, but not shown to be caused by, factors such as speed,
visibility or alcohol, which this data does not record. Crash times were recorded by police and may be
rounded.

### Member 2: AI prompts used
1. "we are working on data engineering project as a team of six and I am taking member two … tell me a
   guide step by step what I need to do and provide me the working code" (Claude, with the Research
   Questions Guide PDF and the team notebook attached)
2. *[add any other prompts you used]*